In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
con =duckdb.connect("../../ProjectData.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

silver_files = sorted(Path("../../data/silver").glob("cleaned_data_load_date=*.parquet"))
if not silver_files:
    raise FileNotFoundError("No Silver Parquet found. Run clean_transform_to_silver.ipynb first.")

silver_path = silver_files[-1]
print(f"Loading Silver from: {silver_path}")

silver_df = pd.read_parquet(silver_path)
silver_df = silver_df.sort_values(["product_parent", "review_date", "_index"]).reset_index(drop=True)
print(f"Rows: {len(silver_df):,}  |  Products: {silver_df['product_parent'].nunique():,}")

In [ ]:
def compute_novelty_for_product(group_df):
    """
    For each review in a product group, compute:
    - product_tfidf_novelty: 1 - cosine_sim(this_review, centroid_of_all_others)
      NaN for single-review products.
    - product_vocab_expansion_ratio: fraction of this review's tokens not seen
      in any chronologically prior review for the same product.
      NaN for the first review (no prior context exists).
    """
    n = len(group_df)
    indices = group_df.index.tolist()
    texts = group_df["review_body"].fillna("").tolist()

    novelty = np.full(n, np.nan)
    vocab_expansion = np.full(n, np.nan)

    if n >= 2:
        try:
            vec = TfidfVectorizer(min_df=1, sublinear_tf=True)
            tfidf_matrix = vec.fit_transform(texts)  
            total = tfidf_matrix.sum(axis=0) 
            for i in range(n):
                others_sum = total - tfidf_matrix[i]  
                centroid = others_sum / (n - 1)
                sim = cosine_similarity(tfidf_matrix[i], centroid)[0, 0]
                novelty[i] = 1.0 - float(sim)
        except Exception:
            pass  

    seen_tokens = set()
    for i, text in enumerate(texts):
        tokens = set(str(text).lower().split())
        if i == 0:
            seen_tokens = tokens
            
        else:
            new_tokens = tokens - seen_tokens
            vocab_expansion[i] = len(new_tokens) / len(tokens) if tokens else np.nan
            seen_tokens |= tokens

    return pd.DataFrame({
        "_index": indices,
        "product_tfidf_novelty": novelty,
        "product_vocab_expansion_ratio": vocab_expansion,
    })


print("Computing within-product TF-IDF novelty and vocab expansion ...")
result_parts = []
for pid, group in silver_df.groupby("product_parent", sort=False):
    result_parts.append(compute_novelty_for_product(group))

novelty_df = pd.concat(result_parts, ignore_index=True)

silver_df = silver_df.reset_index(drop=False).rename(columns={"index": "_row_pos"})
novelty_df = novelty_df.rename(columns={"_index": "_row_pos"})
merged = silver_df[["_row_pos", "_index", "product_id", "product_parent", "label",
                     "product_category_id", "review_date"]].merge(novelty_df, on="_row_pos")

print(f"Done. Novelty NaN rate: {merged['product_tfidf_novelty'].isna().mean():.2%}")
print(f"Vocab expansion NaN rate: {merged['product_vocab_expansion_ratio'].isna().mean():.2%}")
merged[["product_tfidf_novelty", "product_vocab_expansion_ratio"]].describe().round(3)

In [ ]:
con.register("novelty_temp", merged)

con.execute("DROP TABLE IF EXISTS gold.review_novelty_features")
con.execute("""
    CREATE TABLE gold.review_novelty_features AS
    SELECT
        _index,
        product_id,
        product_parent,
        label,
        product_tfidf_novelty,
        product_vocab_expansion_ratio,
        -- Percentile rank of novelty within the product: how novel is this review
        -- relative to the other reviews for the same product?
        PERCENT_RANK() OVER (
            PARTITION BY product_parent
            ORDER BY product_tfidf_novelty NULLS FIRST
        ) AS product_review_rank_novelty
    FROM novelty_temp
    WHERE _index IS NOT NULL
""")

print("gold.review_novelty_features created.")
print("Rows:", con.execute("SELECT COUNT(*) FROM gold.review_novelty_features").fetchone()[0])

In [ ]:
con.execute("""
    SELECT
        COUNT(*)                                                                AS total_rows,
        SUM(CASE WHEN product_tfidf_novelty IS NULL THEN 1 ELSE 0 END)         AS null_novelty,
        SUM(CASE WHEN product_vocab_expansion_ratio IS NULL THEN 1 ELSE 0 END) AS null_vocab_exp,
        ROUND(MIN(product_tfidf_novelty),          3)  AS min_novelty,
        ROUND(MAX(product_tfidf_novelty),          3)  AS max_novelty,
        ROUND(AVG(product_tfidf_novelty),          3)  AS avg_novelty,
        ROUND(AVG(product_vocab_expansion_ratio),  3)  AS avg_vocab_exp,
        ROUND(AVG(product_review_rank_novelty),    3)  AS avg_rank_novelty
    FROM gold.review_novelty_features
""").df()

In [ ]:
con.execute("""
    SELECT
        label,
        COUNT(*)                                            AS reviews,
        ROUND(AVG(product_tfidf_novelty),         3)       AS avg_novelty,
        ROUND(AVG(product_vocab_expansion_ratio), 3)       AS avg_vocab_expansion,
        ROUND(AVG(product_review_rank_novelty),   3)       AS avg_rank_novelty
    FROM gold.review_novelty_features
    WHERE label IS NOT NULL
    GROUP BY label
    ORDER BY label DESC
""").df()

In [ ]:
gold_dir = Path("../../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

output_file = gold_dir / f"novelty_features_load_date={date.today().isoformat()}.parquet"
con.execute(f"""
    COPY (SELECT * FROM gold.review_novelty_features)
    TO '{output_file.as_posix()}' (FORMAT PARQUET)
""")
print(f"Saved: {output_file.resolve()}")

In [ ]:
con.close()